In [1]:
import pandas as pd
import numpy as np
import psutil
import os
import torch
from torch.utils.flop_counter import FlopCounterMode
import time

from neuralforecast.models import NHITS, NHITS_TREAT, NBEATSx, NBEATSx_TREAT, TFT
from ray.tune.search.hyperopt import HyperOptSearch
from neuralforecast.losses.pytorch import HuberLoss
from neuralforecast.core import NeuralForecast

2025-04-16 15:00:05,011	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-04-16 15:00:05,132	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
def get_flops(model, inp, with_backward=False, display=False):
    istrain = model.training
    model.eval()

    flop_counter = FlopCounterMode(mods=model, display=display, depth=None)
    with flop_counter:
        if with_backward:
            model(inp).sum().backward()
        else:
            model(inp)
    total_flops =  flop_counter.get_total_flops()
    if istrain:
        model.train()
    return total_flops

def get_inp(dataset_name, model_name):
    if dataset_name == 'ohiot1dm_exog':
        df = pd.read_csv('../datasets/ohiot1dm_static.csv')
        v = torch.tensor(df.iloc[:4, 1:].values, dtype=torch.float)

        inp = {'insample_y':torch.randn(4, 120),
                   'insample_mask':torch.ones(4, 120),
                    'futr_exog': None, #torch.randn(4, 256, 1),
                    'hist_exog': torch.randn(4, 120, 3),
                    'stat_exog': v,
                    'batch_idx': torch.randn(4, 1),
          }

    elif (dataset_name == 'ohiot1dm')&(model_name!='autotft'):
        df = pd.read_csv('../datasets/ohiot1dm_static.csv')
        v = torch.tensor(df.iloc[:4, 1:].values, dtype=torch.float)

        inp = {'insample_y':torch.randn(4, 120),
                   'insample_mask':torch.ones(4, 120),
                    'futr_exog': [],
                    'hist_exog': [],
                    'stat_exog': v,
          }

    elif (dataset_name == 'ohiot1dm')&(model_name=='autotft'):
        df = pd.read_csv('../datasets/ohiot1dm_static.csv')
        v = torch.tensor(df.iloc[:4, 1:].values, dtype=torch.float)

        inp = {'insample_y':torch.randn(4, 120),
                   'insample_mask':torch.ones(4, 120),
                    'futr_exog': None,
                    'hist_exog': None,
                    'stat_exog': v,
                    'batch_idx': torch.randn(4, 1),
          }

    return inp

def count_parameters(model):
    total_params = 0
    for p in model.parameters():
        if p.requires_grad:
            total_params += p.numel()
    return total_params

def get_inference_time(model, inp, num_runs=100):
    # Ensure model is in evaluation mode
    model.eval()
    num_runs = 100
    
    # # Measure inference time
    # with torch.no_grad():  # Disable gradients for faster execution
    #     start_time = time.perf_counter()
    #     model(input_tensor)  # Single forward pass
    #     end_time = time.perf_counter()
    
    # # Compute elapsed time in milliseconds
    # inference_time = (end_time - start_time) * 1000
    # print(f"CPU Inference Time: {inference_time:.3f} ms") 
    
    times = []
    for run in range(num_runs):
        with torch.no_grad():  # Disable gradients for faster execution
            start_time = time.perf_counter()
            model(inp)  # Single forward pass
            end_time = time.perf_counter()
        
            times.append((end_time - start_time) * 1000)
    
    # Compute average inference time
    avg_time = np.mean(np.array(times))

    return avg_time

# Function to get the current memory usage in MB
def get_cpu_memory_usage():
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    return memory_info.rss / (1024 ** 2)  # Convert to MB

def get_mem(model, inp):
    # Create the model and move it to the CPU
    device = torch.device("cpu")
    model = model.to(device)
    
    # Track memory usage before inference
    initial_memory = get_cpu_memory_usage()
    #print(f"Initial CPU Memory Usage (MB): {initial_memory}")
    output = model(inp)
    final_memory = get_cpu_memory_usage()
    #print(f"Final CPU Memory Usage (MB): {final_memory}")
    #print(f"Memory used for inference (MB): {final_memory - initial_memory}")

    return final_memory - initial_memory


In [3]:
def get_model(model_name, dataset_name, hparams):

    if model_name == 'autonhits':
        model = NHITS(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    rrandom_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    stack_types = ['identity', 'identity', 'identity'],
                    n_blocks = [1, 1, 1],
                    mlp_units = [[1024, 1024], [1024, 1024], [1024, 1024]],
                    n_pool_kernel_size = [1, 1, 1],
                    n_freq_downsample = [1, 1, 1],
                    dropout_prob_theta = 0.0,
                     )
    
    elif model_name == 'autonhitstreat':
        model = NHITS_TREAT(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    random_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    stack_types = hparams['stack_types'],
                    n_blocks = [1, 1, 1],
                    mlp_units = [[1024, 1024], [1024, 1024], [1024, 1024]],
                    n_pool_kernel_size = [1, 1, 1],
                    n_freq_downsample = [1, 1, 1],
                    dropout_prob_theta = 0.0,
                    concentrator_type = hparams['concentrator_type'],
                    n_series = hparams['n_series'],
                    init_ka1 = 1.5,
                    init_ka2 = 1.5,
                    init_ka3 = 1.5,
                    freq = 5
                     )
        
    elif model_name == 'autonbeatsx':
        model = NBEATSx(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    random_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    n_harmonics = 2,
                    n_polynomials = 2,
                    stack_types = ['identity', 'trend', 'seasonality'],
                    n_blocks = [1, 1, 1],
                    mlp_units = [[1024, 1024], [1024, 1024], [1024, 1024]],
                    dropout_prob_theta = 0.0,
                     )

    elif model_name == 'autonbeatsxtreatcts':
        model = NBEATSx_TREAT(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    random_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    n_harmonics = 2,
                    n_polynomials = 2,
                    stack_types = ['concentrator', 'trend', 'seasonality'],
                    n_blocks = [1, 1, 1],
                    mlp_units = [[1024, 1024], [1024, 1024], [1024, 1024]],
                    dropout_prob_theta = 0.0,
                    concentrator_type = hparams['concentrator_type'],
                    n_series = hparams['n_series'],
                    init_ka1 = 1.5,
                    init_ka2 = 1.5,
                    init_ka3 = 1.5,
                    freq = 5
                     )

    elif model_name == 'autotft':
        if 'init_ka3' not in hparams.keys():
            print('no init_ka3')
            model = TFT(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    random_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    hidden_size = hparams['hidden_size'],
                    n_head = hparams['n_head'],
                    attn_dropout = 0.0,
                    dropout = 0.0,
                    tgt_size = hparams['tgt_size'],
                    use_concentrator = hparams['use_concentrator'],
                    concentrator_type = hparams['concentrator_type'],
                    n_series = hparams['n_series'],
                    init_ka1 = hparams['init_ka1'],
                    init_ka2 = hparams['init_ka1'],
                    #init_ka3 = hparams['init_ka3'],
                    freq = hparams['freq'],
                     )
        else:
            model = TFT(h = 6,
                    input_size = 120,
                    loss = HuberLoss(),
                    valid_loss= HuberLoss(),
                    learning_rate = hparams['learning_rate'],
                    max_steps = 2000,
                    val_check_steps = 100,
                    batch_size = 4,
                    valid_batch_size = None,
                    windows_batch_size = 256,
                    inference_windows_batch_size = -1,
                    step_size = 1,
                    num_lr_decays = 3,
                    early_stop_patience_steps = 5,
                    scaler_type = hparams['scaler_type'],
                    stat_exog_list = hparams['stat_exog_list'],
                    hist_exog_list = hparams['hist_exog_list'],
                    futr_exog_list = hparams['futr_exog_list'],
                    random_seed = hparams['random_seed'],
                    alias = hparams['alias'],
                    hidden_size = hparams['hidden_size'],
                    n_head = hparams['n_head'],
                    attn_dropout = 0.0,
                    dropout = 0.0,
                    tgt_size = hparams['tgt_size'],
                    use_concentrator = hparams['use_concentrator'],
                    concentrator_type = hparams['concentrator_type'],
                    n_series = hparams['n_series'],
                    init_ka1 = hparams['init_ka1'],
                    init_ka2 = hparams['init_ka1'],
                    init_ka3 = hparams['init_ka3'],
                    freq = hparams['freq'],
                     )

    return model 


In [4]:

paths = [ 
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_baselines/results_baselineNHITS/ohiot1dm_6/baseline_models/trial_baseline_0/autonhits_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_baselines/results_baselineNHITS/ohiot1dm_exog_6/baseline_models/trial_baseline_0/autonhits_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_sumtotal_nhits/ohiot1dm_exog_6/sum_total_models/trial_sum_total_0/autonhitstreat_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/CHIL2025_rebuttal_results/6horizon120contextlenforflop/ohiot1dm_exog_6/treat_models/trial_treat_0/autonhitstreat_0.ckpt',
    
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_6/baseline_models/trial_baseline_0/autonbeatsx_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_exog_6/baseline_models/trial_baseline_0/autonbeatsx_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_sumtotal_nbeats_tft/ohiot1dm_exog_6/sum_total_models/trial_sum_total_0/autonbeatsxtreatcts_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_exog_6/treat_models/trial_treat_0/autonbeatsxtreatcts_0.ckpt',
    
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_6/baseline_models/trial_baseline_0/autotft_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_exog_6/baseline_models/trial_baseline_0/autotft_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_sumtotal_nbeats_tft/ohiot1dm_exog_6/sum_total_models/trial_sum_total_0/autotft_0.ckpt',
    '/home/extra_scratch/wpotosna/CHIL2025_results/results_nbeats_dlinear_tft/ohiot1dm_exog_6/treat_models/trial_treat_0/autotft_0.ckpt',
]
dataset_names = [
    'ohiot1dm',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
    'ohiot1dm',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
    'ohiot1dm',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
    'ohiot1dm_exog',
]

names = [
    'NHITS',
    'NHITS Exog.',
    'NHITS Sum-Total',
    'NHITS PK',
    'NBEATSx',
    'NBEATSx Exog.',
    'NBEATSx Sum-Total',
    'NBEATSx PK',
    'TFT',
    'TFT Exog.',
    'TFT Sum-Total',
    'TFT PK',
]

model_flop_count = {}
model_num_param = {}
inference_time = {}
mem = {}
for path, dataset_name, name in zip(paths, dataset_names, names):
    print(name)
    checkpoint = torch.load(path)
    state_dict = checkpoint['state_dict']
    hparams = checkpoint['hyper_parameters']
    
    model_name = path.split('/')[-1]
    model_name = model_name.split('_')[0]
    model = get_model(model_name, dataset_name, hparams)
    inp = get_inp(dataset_name, model_name)
    model.load_state_dict(state_dict)
    
    model_flop_count[name] = get_flops(model, inp)
    model_num_param[name] = count_parameters(model)
    inference_time[name] = get_inference_time(model, inp=inp)
    mem[name] = get_mem(model, inp)


NHITS


Seed set to 1
Seed set to 1


NHITS Exog.


Seed set to 2


NHITS Sum-Total


...

Seed set to 7


no init_ka3
TFT Sum-Total


Seed set to 3


TFT PK


Seed set to 8


In [6]:
model_flop_count

{'NHITS': 81936384,
 'NHITS Exog.': 90783744,
 'NHITS Sum-Total': 84885504,
 'NHITS PK': 84885504,
 'NBEATSx': 80098096,
 'NBEATSx Exog.': 88945456,
 'NBEATSx Sum-Total': 83047216,
 'NBEATSx PK': 83047216,
 'TFT': 365524992,
 'TFT Exog.': 611334144,
 'TFT Sum-Total': 605140992,
 'TFT PK': 607205376}

In [13]:
model_num_param

{'NHITS': 10254714,
 'NHITS Exog.': 11360634,
 'NHITS Sum-Total': 10623390,
 'NHITS PK': 10623390,
 'NBEATSx': 10023064,
 'NBEATSx Exog.': 11128984,
 'NBEATSx Sum-Total': 10391740,
 'NBEATSx PK': 10391740,
 'TFT': 2526302,
 'TFT Exog.': 2784749,
 'TFT Sum-Total': 2778605,
 'TFT PK': 2780689}

In [19]:
inference_time

{'NHITS': 8.320982409641147,
 'NHITS Exog.': 9.232339662266895,
 'NHITS Sum-Total': 9.733813761267811,
 'NHITS PK': 12.48242444708012,
 'NBEATSx': 7.461662223795429,
 'NBEATSx Exog.': 8.772294576046988,
 'NBEATSx Sum-Total': 8.35784034919925,
 'NBEATSx PK': 10.374626696575433,
 'TFT': 26.920248571550474,
 'TFT Exog.': 29.498751463834196,
 'TFT Sum-Total': 33.82124426658265,
 'TFT PK': 31.972548629855737}

In [20]:
mem

{'NHITS': 0.0859375,
 'NHITS Exog.': 0.00390625,
 'NHITS Sum-Total': 0.0,
 'NHITS PK': 0.0,
 'NBEATSx': 0.0,
 'NBEATSx Exog.': 0.0,
 'NBEATSx Sum-Total': 0.0,
 'NBEATSx PK': 0.89453125,
 'TFT': 7.55078125,
 'TFT Exog.': 17.140625,
 'TFT Sum-Total': 26.01171875,
 'TFT PK': 18.29296875}